In [8]:
# препроцессинг 
import pandas as pd
import numpy as np
import re
import html
import unicodedata
import nltk


# загрузка стоп-слов и лемм 

nltk.pathsec.ALLOW_PROXIED_FETCH = True

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import (
    WordNetLemmatizer,
    PorterStemmer
)

# ML Модели: 
# =====================================

# векторизация 
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer


# модели классификации 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# понижение размерности 
from sklearn.decomposition import TruncatedSVD

# метрики качества 
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# формирование train/test 
from sklearn.model_selection import train_test_split

# класс DataSet 
from datasets import Dataset


# Fine-Tuning предобученной модели 
#========================================

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df_train = pd.read_csv('../data/train.csv')

df_test = pd.read_csv('../data/test.csv')

In [3]:
df_train.shape 

(7613, 5)

In [4]:
df_train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
# example = df_train[~df_train['keyword'].isna()].iloc[0:10, [0,3]]

<hr>

1. Формирование версии датасета 

In [6]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Отрицания оставляем
stop_words -= {"no", "not", "nor"}

In [7]:
def preprocess_text(
    text,
    remove_stopwords=False,
    normalization=None,
    lowercase=True
):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)

    # Замена ссылок и упоминаний
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"(?<!\w)@[A-Za-z0-9_]+", " USER ", text)

    # Убираем #, но сохраняем слово
    text = re.sub(r"#(\w+)", r"\1", text)

    # Удаляем токены, содержащие числа
    text = re.sub(r"\b\w*\d\w*\b", " ", text)

    # Удаляем лишние символы
    text = re.sub(r"[^\w\s']", " ", text)

    if lowercase:
        text = text.lower()

    # Разбиваем на слова
    tokens = text.split()

    # Удаление стоп-слов
    if remove_stopwords:
        tokens = [
            word for word in tokens
            if word.lower() not in stop_words
        ]

    # Лемматизация
    if normalization == "lemma":
        tokens = [
            lemmatizer.lemmatize(word)
            for word in tokens
        ]

    # Стемминг
    elif normalization == "stem":
        tokens = [
            stemmer.stem(word)
            for word in tokens
        ]

    return " ".join(tokens)

In [8]:
versions = {
    # удаление спец символов/пунктуации + lowercase 
    "v1_basic": {},
    # удаление спец символов + lowercase + stopwords 
    "v2_no_stopwords": {
        "remove_stopwords": True
    },
    # удаление спец символов/пунктуации + lowercase + лемматизация 
    "v3_lemma": {
        "normalization": "lemma"
    },
    # удаление спец символов/пунктуации + lowercase + stopwords + лемматизация
    "v4_lemma_no_stopwords": {
        "normalization": "lemma",
        "remove_stopwords": True
    },
    # удаление спец символов/пунктуации + lowercase + стемминг 
    "v5_stem": {
        "normalization": "stem"
    },
    # только удаление спец символов/пунктуации 
    "v6_original_case": {
        "lowercase": False
    }
}

df_versions_train = df_train.copy()
df_versions_test = df_test.copy()


for name, params in versions.items():
    df_versions_train[name] = df_train["text"].apply(
        lambda text: preprocess_text(text, **params)
    )
    
for name, params in versions.items():
    df_versions_test[name] = df_test["text"].apply(
        lambda text: preprocess_text(text, **params)
    )

Важно: после очистки твитов могут появляться дубли. В зависимости от способа их разное количество и это влияет на итоговую оценку качества. 

In [9]:
# df_versions_train.to_csv('../data/train_preprocessing_versions.csv')

<hr>

2. TF-IDF

In [10]:
tfidf_dataset = {}
vectorizers = {}

y = df_versions_train["target"]

train_idx, val_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

for version in versions: 
    
    vectorizer = TfidfVectorizer(
        analyzer="word",
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        min_df=2,
        max_df=0.95,
        max_features=10000,
        norm="l2",
        sublinear_tf=True
    )
    
    X_train = vectorizer.fit_transform(df_versions_train[version].iloc[train_idx])
    X_val = vectorizer.transform(df_versions_train[version].iloc[val_idx])
    X_test = vectorizer.transform(df_versions_test[version])
    
    vectorizers[version] = vectorizer
    
    tfidf_dataset[version] = {
        'X_train': X_train, 
        'X_val': X_val, 
        'X_test': X_test
    }

In [11]:
tfidf_dataset['v3_lemma']['X_test'][0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (1, 5063)>

<hr>

3. Classic Ml Классификация. Logistic Regression / Gradient Boosting 

In [12]:
results = []


for version in versions:

    print(f"Обрабатывается версия: {version}")

    # TF-IDF матрицы
    X_train = tfidf_dataset[version]["X_train"]
    X_val = tfidf_dataset[version]["X_val"]
    X_test = tfidf_dataset[version]["X_test"]

    # =====================================
    # 1. Logistic Regression
    # =====================================

    lr = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    lr.fit(X_train, y_train)
    pred_lr_val = lr.predict(X_val)
    pred_lr_test = lr.predict(X_test)

    results.append({
        "version": version,
        "model": "Logistic Regression", 
        "accuracy": accuracy_score(y_val, pred_lr_val),
        "precision": precision_score(y_val, pred_lr_val, zero_division=0),
        "recall": recall_score(y_val, pred_lr_val, zero_division=0),
        "f1": f1_score(y_val, pred_lr_val, zero_division=0),
        "predictions_val": pred_lr_val,
        "predictions_test": pred_lr_test,
    })

    # =====================================
    # 2. Gradient Boosting
    # =====================================

    # Снижаем размерность разреженных TF-IDF-векторов
    n_components = min(
        100,
        X_train.shape[1],
        X_train.shape[0] - 1
    )

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=42
    )

    X_train_reduced = svd.fit_transform(X_train)
    X_val_reduced = svd.transform(X_val)
    X_test_reduced = svd.transform(X_test)

    gb = GradientBoostingClassifier(
        random_state=42
    )

    gb.fit(X_train_reduced, y_train)
    pred_gb_val = gb.predict(X_val_reduced)
    pred_gb_test = gb.predict(X_test_reduced)

    results.append({
        "version": version,
        "model": "Gradient Boosting",
        "accuracy": accuracy_score(y_val, pred_gb_val),
        "precision": precision_score(y_val, pred_gb_val, zero_division=0),
        "recall": recall_score(y_val, pred_gb_val, zero_division=0),
        "f1": f1_score(y_val, pred_gb_val, zero_division=0),
        "predictions_val": pred_gb_val,
        "predictions_test": pred_gb_test,
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

Обрабатывается версия: v1_basic
Обрабатывается версия: v2_no_stopwords
Обрабатывается версия: v3_lemma
Обрабатывается версия: v4_lemma_no_stopwords
Обрабатывается версия: v5_stem
Обрабатывается версия: v6_original_case


In [13]:
X_test.shape 

(3263, 6570)

In [14]:
results_df

,version,model,accuracy,precision,recall,f1,predictions_val,predictions_test
0,v1_basic,Logistic Regression,0.819435,0.828423,0.730887,0.776604,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,v3_lemma,Logistic Regression,0.818122,0.825561,0.730887,0.775345,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,v5_stem,Logistic Regression,0.817466,0.830986,0.721713,0.772504,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,v4_lemma_no_stopwords,Logistic Regression,0.816809,0.837838,0.711009,0.769231,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,v2_no_stopwords,Logistic Regression,0.816152,0.835125,0.712538,0.768977,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,v6_original_case,Logistic Regression,0.811556,0.834244,0.700306,0.761430,"[1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, ...","[1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,v4_lemma_no_stopwords,Gradient Boosting,0.776756,0.787546,0.657492,0.716667,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, ...","[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,v2_no_stopwords,Gradient Boosting,0.768221,0.777164,0.645260,0.705096,"[1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, ...","[1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,v1_basic,Gradient Boosting,0.758372,0.760000,0.639144,0.694352,"[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
9,v5_stem,Gradient Boosting,0.755089,0.764595,0.620795,0.685232,"[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [15]:
results_df['predictions_test'][0].shape, df_test.shape 

((3263,), (3263, 4))

In [16]:
def create_report(results_df):
    
    for index, row in results_df.iterrows():
        
        predictions = pd.Series(row['predictions_test'], name='target')
        
        report_df = pd.concat([df_test['id'].reset_index(drop=True), predictions.reset_index(drop=True)], axis=1)
        
        report_df.to_csv(f"../data/reports/{row['version']}_{row['model']}.csv", index=False)
    

In [17]:
create_report(results_df)

<hr>

4. Предобученная модель + Classic Ml классификация

In [18]:
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

embeddings = embedder.encode(
    df_train["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(embeddings.shape)  # (7613, 768)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/238 [00:00<?, ?it/s]

(7613, 768)


In [19]:
X_train_emb = embeddings[train_idx]
X_val_emb = embeddings[val_idx]

In [20]:
X_test_emb = embedder.encode(
    df_test["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(embeddings.shape)  # (7613, 768)

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

(7613, 768)


In [21]:
lr_emb = LogisticRegression(max_iter=1000, random_state=42)
lr_emb.fit(X_train_emb, y_train)

pred_val = lr_emb.predict(X_val_emb)
pred_test = lr_emb.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_val, pred_val))
print("Precision:", precision_score(y_val, pred_val, zero_division=0))
print("Recall:", recall_score(y_val, pred_val, zero_division=0))
print("F1:", f1_score(y_val, pred_val, zero_division=0))

Accuracy: 0.8365068942875903
Precision: 0.8314238952536824
Recall: 0.7767584097859327
F1: 0.8031620553359684


In [22]:
result = pd.DataFrame({
    "id": df_test["id"].to_numpy(),
    "target": pred_test,
})

result.to_csv("../data/reports/bge-base-en-v1.5.csv", index=False)

<hr>

5. Fine-tuning предобученного трансформера

Задача: по тексту сообщения предсказать `target` (0 или 1).
В пункте 4 мы получали готовые эмбеддинги и обучали Logistic Regression.
Здесь дообучаем **все веса трансформера вместе с новым слоем классификации**
на размеченных сообщениях.

5.1. Выбор модели

Используем [distilbert/distilbert-base-uncased](https://huggingface.co/distilbert/distilbert-base-uncased).
Это англоязычный encoder-трансформер, полученный дистилляцией BERT:
он компактнее BERT, поэтому удобен для первого эксперимента с ограниченными ресурсами.
Язык модели соответствует английским текстам нашего датасета.
Берём базовую предобученную модель и добавляем классификатор с двумя выходами.
Это разумная стартовая модель; превосходство над предыдущими подходами проверяем по метрикам.

In [ ]:
model_name = "distilbert/distilbert-base-uncased"
seed = 42
max_length = 128
set_seed(seed)

print("Устройство:", "CUDA GPU" if torch.cuda.is_available() else "CPU")

c:\Anaconda\envs\py311\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Устройство: CUDA GPU


5.2. Данные и предобработка

Берём исходный столбец `text`, а не варианты после стемминга или лемматизации.
Удаление стоп-слов, отрицаний и пунктуации может убрать полезный контекст.
Для этого эксперимента оставляем также числа, ссылки, упоминания и хештеги.
Исправляем только HTML-сущности (`&amp;` → `&`) и лишние пробелы; пропуски заменяем пустой строкой.
Регистр обработает токенизатор uncased-модели, вручную вызывать `lower()` не нужно.

Токенизатор переводит текст в `input_ids` и `attention_mask`.
Ограничиваем длину 128 токенами как стартовую настройку для коротких сообщений;
ниже проверяем, какая доля текстов будет обрезана.
Padding выполняется до длины самого длинного текста **в текущем батче**.

Как и раньше, выделяем стратифицированную validation-выборку 20% с `random_state=42`.
В `test.csv` нет `target`: метрики считаем на validation, для test сохраняем только предсказания.

In [3]:
data_dir = Path("../data")
transformer_train_df = pd.read_csv(data_dir / "train.csv")
transformer_test_df = pd.read_csv(data_dir / "test.csv")

assert transformer_train_df["target"].notna().all()
assert set(transformer_train_df["target"].unique()) == {0, 1}

transformer_train_idx, transformer_val_idx = train_test_split(
    np.arange(len(transformer_train_df)),
    test_size=0.2,
    random_state=seed,
    stratify=transformer_train_df["target"],
)

def normalize_transformer_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

def make_transformer_dataset(frame, with_labels=True):
    data = {"text": frame["text"].map(normalize_transformer_text).tolist()}
    if with_labels:
        data["labels"] = frame["target"].astype("int64").tolist()
    return Dataset.from_dict(data).map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )

transformer_train_ds = make_transformer_dataset(
    transformer_train_df.iloc[transformer_train_idx]
)
transformer_val_ds = make_transformer_dataset(
    transformer_train_df.iloc[transformer_val_idx]
)
transformer_test_ds = make_transformer_dataset(
    transformer_test_df, with_labels=False
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

token_lengths = tokenizer(
    transformer_train_df["text"].map(normalize_transformer_text).tolist(),
    truncation=False,
)["input_ids"]
truncated_share = np.mean([len(ids) > max_length for ids in token_lengths])
print(f"Доля train-текстов длиннее {max_length} токенов: {truncated_share:.2%}")
print("Train / validation / test:", len(transformer_train_ds),
      len(transformer_val_ds), len(transformer_test_ds))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

Доля train-текстов длиннее 128 токенов: 0.00%
Train / validation / test: 6090 1523 3263


5.3. Обучение через transformers.Trainer

Добавляем к DistilBERT слой классификации на два класса.
Предупреждение о новых, случайно инициализированных весах классификатора ожидаемо:
этот слой как раз нужно обучить на нашей задаче.

Стартовые параметры: 3 эпохи, learning rate `2e-5`, batch size 8,
накопление градиентов за 2 шага (эффективный batch size 16 на одном устройстве).
Маленький learning rate позволяет постепенно адаптировать предобученные веса.
После каждой эпохи считаем метрики и сохраняем checkpoint;
в конце загружаем checkpoint с наибольшим validation F1.
Веса encoder не замораживаем, поэтому это fine-tuning всей модели.

In [4]:
def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }

set_seed(seed)
transformer_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "NOT_DISASTER", 1: "DISASTER"},
    label2id={"NOT_DISASTER": 0, "DISASTER": 1},
)

model_dir = Path("../models/distilbert-disaster")
training_args = TrainingArguments(
    output_dir=str(model_dir / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    optim="adamw_torch",
    dataloader_num_workers=0,
    seed=seed,
    data_seed=seed,
    report_to="none",
    push_to_hub=False,
)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=transformer_train_ds,
    eval_dataset=transformer_val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_transformer_metrics,
)
print("Обучаемых параметров:", sum(
    p.numel() for p in transformer_model.parameters() if p.requires_grad
))

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Обучаемых параметров: 66955010


In [5]:
# Эта ячейка запускает обучение; время зависит от оборудования.
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.929712,0.379197,0.845043,0.864111,0.758410,0.807818
2,0.713910,0.402386,0.836507,0.809160,0.810398,0.809778
3,0.465898,0.435351,0.833224,0.805810,0.805810,0.805810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

5.4. Метрики и сравнение

Получаем предсказания лучшего checkpoint на validation.
Accuracy — доля правильных ответов; precision — точность предсказаний класса 1;
recall — доля найденных сообщений класса 1; F1 — гармоническое среднее precision и recall.
Выбираем модель по F1, а остальные метрики помогают понять характер ошибок.
В confusion matrix строки соответствуют истинным классам, столбцы — предсказанным.

Сравни эту таблицу с TF-IDF и BGE на том же разбиении.
Поскольку validation используется для выбора checkpoint, это оценка для выбора модели,
а не независимая финальная test-оценка. Численные выводы делаем после выполнения обучения.

In [6]:
val_output = trainer.predict(transformer_val_ds, metric_key_prefix="val")
transformer_val_predictions = val_output.predictions.argmax(axis=-1)

transformer_metrics_df = pd.DataFrame([{
    "model": "DistilBERT fine-tuned",
    **{name: val_output.metrics[f"val_{name}"]
       for name in ("accuracy", "precision", "recall", "f1")},
}])
print(transformer_metrics_df.to_string(index=False))
print("Лучший checkpoint:", trainer.state.best_model_checkpoint)
print(classification_report(
    val_output.label_ids,
    transformer_val_predictions,
    labels=[0, 1],
    target_names=["not_disaster", "disaster"],
    digits=4,
    zero_division=0,
))
print("Confusion matrix [0, 1]:")
print(confusion_matrix(
    val_output.label_ids, transformer_val_predictions, labels=[0, 1]
))

                model  accuracy  precision   recall       f1
DistilBERT fine-tuned  0.836507    0.80916 0.810398 0.809778
Лучший checkpoint: ..\models\distilbert-disaster\checkpoints\checkpoint-762
              precision    recall  f1-score   support

not_disaster     0.8571    0.8562    0.8566       869
    disaster     0.8092    0.8104    0.8098       654

    accuracy                         0.8365      1523
   macro avg     0.8332    0.8333    0.8332      1523
weighted avg     0.8365    0.8365    0.8365      1523

Confusion matrix [0, 1]:
[[744 125]
 [124 530]]


5.5. Предсказания test и сохранение

Сохраняем предсказания в исходном порядке `test.csv`, таблицу validation-метрик
и лучшую модель вместе с токенизатором. Повторный запуск перезапишет эти файлы.
Метрик для test здесь нет: истинные ответы в нём отсутствуют.

При нехватке GPU-памяти уменьши `per_device_train_batch_size` до 4
и увеличь `gradient_accumulation_steps` до 4, чтобы сохранить эффективный размер батча.
Если нет GPU, можно перенести ноутбук и папку `data/` в Colab с GPU;
сохрани расположение `notebooks/` и `data/`, чтобы относительные пути работали.

Методика: [Hugging Face — Text classification](https://huggingface.co/docs/transformers/tasks/sequence_classification).

In [7]:
test_output = trainer.predict(transformer_test_ds)
transformer_test_predictions = test_output.predictions.argmax(axis=-1)
transformer_submission = pd.DataFrame({
    "id": transformer_test_df["id"].to_numpy(),
    "target": transformer_test_predictions.astype(int),
})
assert len(transformer_submission) == len(transformer_test_df)
assert transformer_submission["id"].tolist() == transformer_test_df["id"].tolist()
assert transformer_submission["target"].isin([0, 1]).all()

reports_dir = data_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
transformer_submission.to_csv(reports_dir / "distilbert-finetuned.csv", index=False)
transformer_metrics_df.to_csv(
    reports_dir / "distilbert-finetuned-validation-metrics.csv", index=False
)
trainer.save_model(str(model_dir / "best"))
tokenizer.save_pretrained(str(model_dir / "best"))
print("Предсказания:", reports_dir / "distilbert-finetuned.csv")
print("Модель и токенизатор:", model_dir / "best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Предсказания: ..\data\reports\distilbert-finetuned.csv
Модель и токенизатор: ..\models\distilbert-disaster\best
